In [67]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict, Literal, Annotated
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage ,BaseMessage
from langgraph.checkpoint.memory import MemorySaver
import operator

In [68]:
from langgraph.graph.message import add_messages
class ChatState(TypedDict):
    message: Annotated[list[BaseMessage], add_messages]


In [69]:
llm = ChatOpenAI()
def chat_node(state:ChatState):
    #take  user queryt from state
    message = state['message']
    #send to llm
    response = llm.invoke(message)
    #response store to state
    return {"message":[response]}


In [70]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)
#add nodes
graph.add_node("chat_node",chat_node)

graph.add_edge(START ,'chat_node')
graph.add_edge('chat_node',END)
chatbot = graph.compile(checkpointer=checkpointer)


In [71]:
intial_state = {
    'message':[HumanMessage(content="What is stealth Address om monero")]
}


In [72]:
thread_id = "1"

while True:
    user_message = input("Type here: ")

    print("User:", user_message)

    if user_message.strip().lower() in ["exit", "quit", "bye"]:
        break

    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    response = chatbot.invoke(
        {
            "message": [
                HumanMessage(content=user_message)
            ]
        },
        config=config
    )

    print("AI:", response["message"][-1].content)t

User: my name is everes
AI: Hello Everes! How can I assist you today?
User: tell me my name
AI: Your name is Everes.
User: what is zcash
AI: Zcash is a cryptocurrency that focuses on privacy and anonymity. It uses advanced cryptographic techniques such as zero-knowledge proofs to ensure the security and privacy of transactions on its network. Zcash transactions can be shielded or transparent, giving users the option to protect their privacy while using the cryptocurrency.
User:   how it different form other
AI: Zcash is unique compared to other cryptocurrencies because of its focus on privacy and anonymity. It offers users the option to shield their transactions using advanced cryptographic techniques, such as zero-knowledge proofs, which allow parties to prove the validity of a statement without revealing any additional information. This feature sets Zcash apart from other cryptocurrencies that may not prioritize privacy to the same extent. Additionally, Zcash has a fixed supply cap o